<a href="https://colab.research.google.com/github/Tetbet/AQI_Predictor_BYOP/blob/main/notebooks/02_AQI_ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

# 1. Load the cleaned data from your GitHub
url = 'https://raw.githubusercontent.com/Tetbet/AQI_Predictor_BYOP/main/city_day.csv'
df = pd.read_csv(url)

# Clean up any remaining NaNs (Machine Learning cannot handle NaNs at all)
df = df.dropna(subset=['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3', 'AQI', 'AQI_Bucket'])

# 2. Define our Features (X) and Targets (y)
X = df[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']] # The pollutants
y_reg = df['AQI']                                   # Target for Regression (Exact Number)
y_clf = df['AQI_Bucket']                            # Target for Classification (Category Text)

# 3. Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

print("Data successfully split for training!")

Data successfully split for training!


In [2]:
# 4. REGRESSION: Predict the exact AQI number
print("--- Training Regression Model ---")
reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train, y_train_reg)

reg_predictions = reg_model.predict(X_test)
mse = mean_squared_error(y_test_reg, reg_predictions)
print(f"Regression Mean Squared Error: {mse:.2f}")

# Test the regression model with fake sensor data (PM2.5, PM10, NO2, CO, SO2, O3)
sample_data = [[120.5, 140.2, 45.1, 1.2, 12.4, 34.1]]
predicted_aqi = reg_model.predict(sample_data)
print(f"Predicted AQI for sample data: {predicted_aqi[0]:.2f}")

--- Training Regression Model ---
Regression Mean Squared Error: 828.93
Predicted AQI for sample data: 238.82


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [3]:
# 5. CLASSIFICATION: Predict the Health Category
print("\n--- Training Classification Model ---")
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train, y_train_clf)

clf_predictions = clf_model.predict(X_test)
acc = accuracy_score(y_test_clf, clf_predictions)
print(f"Classification Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_clf, clf_predictions))


--- Training Classification Model ---
Classification Accuracy: 81.29%

Classification Report:
              precision    recall  f1-score   support

        Good       0.80      0.69      0.74       234
    Moderate       0.82      0.85      0.83      1117
        Poor       0.73      0.70      0.71       312
Satisfactory       0.83      0.85      0.84      1216
      Severe       0.83      0.77      0.80       100
   Very Poor       0.78      0.72      0.75       223

    accuracy                           0.81      3202
   macro avg       0.80      0.76      0.78      3202
weighted avg       0.81      0.81      0.81      3202



In [4]:
!pip install gradio

In [5]:
import gradio as gr

# 1. Define the function that the UI will use
def predict_aqi(pm25, pm10, no2, co, so2, o3):
    # Format the input data for our ML models
    input_data = [[pm25, pm10, no2, co, so2, o3]]

    # Get the exact number from the Regression model
    predicted_number = reg_model.predict(input_data)[0]

    # Get the text category from the Classification model
    predicted_category = clf_model.predict(input_data)[0]

    # Create a custom health warning based on the category (Simulating our Prolog rules!)
    if predicted_category in ['Poor', 'Very Poor', 'Severe']:
        warning = "⚠️ WARNING: Unsafe air quality. Please wear a mask and avoid outdoor activities."
    else:
        warning = "✅ Air quality is acceptable. You are good to go outside!"

    return round(predicted_number, 2), predicted_category, warning

# 2. Create the visual Interface
interface = gr.Interface(
    fn=predict_aqi,
    inputs=[
        gr.Number(label="PM2.5 (e.g., 45.5)", value=45.5),
        gr.Number(label="PM10 (e.g., 82.1)", value=82.1),
        gr.Number(label="NO2 (e.g., 20.4)", value=20.4),
        gr.Number(label="CO (e.g., 1.2)", value=1.2),
        gr.Number(label="SO2 (e.g., 10.5)", value=10.5),
        gr.Number(label="O3 (e.g., 34.2)", value=34.2)
    ],
    outputs=[
        gr.Textbox(label="Predicted AQI (Number)"),
        gr.Textbox(label="Predicted Category"),
        gr.Textbox(label="Health Advisory")
    ],
    title="🌍 AI Air Quality Predictor",
    description="Enter the pollutant levels below. The AI will use Random Forest Regression to predict the exact AQI and Classification to determine the health category."
)

# 3. Launch the UI!
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://12d68a9c16a57f5fbe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
